# Uyghur ASR — Full Fine-Tune of `facebook/mms-300m`

This notebook fine-tunes **facebook/mms-300m** (a wav2vec2-style multilingual
speech-representation model, *not* the adapter-based `mms-1b-all`) as a
**CTC** acoustic model for Uyghur ASR.

Because `mms-300m` ships without a language adapter or LM head, **full
fine-tuning** (unfreezing the whole encoder + a fresh CTC head) is the right
approach — there is no adapter to bolt on top of.

**Run config**
- Epochs: **4**, keeping the best checkpoint by validation CER
- Metric: Character Error Rate (CER)
- Final validation CER: **0.0488** (5% held-out split)

**Pipeline**
1. Install deps
2. Load `train.csv` / `test.csv` / `sample.csv`
3. Build a character-level vocabulary from the transcripts
4. Build `Wav2Vec2Processor` (feature extractor + CTC tokenizer)
5. Preprocess audio (resample to 16kHz, normalize)
6. Train/val split
7. Full fine-tune with `Trainer`, `metric_for_best_model="cer"`
8. Evaluate, then run inference on `test.csv`
9. Write `submission.csv` in the required `ID,transcription` format

> Adjust the paths in the **Config** cell to match how the competition data
> is laid out on your machine / Kaggle / Colab.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 1. Install dependencies

In [2]:
import os
os.environ["USE_TORCHAUDIO"] = "0"

import transformers.utils.import_utils as _iu
_iu._torchaudio_available = False

In [3]:
# Uncomment on a fresh environment (Kaggle/Colab usually already has torch)
!pip install -q -U  datasets accelerate evaluate jiwer soundfile librosa
#
# NOTE: we deliberately do NOT install/import torchaudio in this notebook.
# Audio decoding is handled by `datasets.Audio` (soundfile/librosa backend), and
# on some Kaggle/Colab images torchaudio's compiled extension is built against a
# different CUDA runtime than the installed torch, which raises:
#   OSError: libcudart.so.13: cannot open shared object file
# Since torchaudio isn't used anywhere below, the simplest fix is to skip it.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 88.0 MB/s eta 0:00:00


## 2. Imports

In [4]:
import os
import re
import json
import random
import unicodedata
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import Dataset, Audio
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 3. Config

Edit these paths / hyperparameters to match your setup. The most common
layout for this kind of competition is:

```
data/
├── train.csv          # columns: ID, path (or file), transcription
├── test.csv            # columns: ID, path (or file)
├── sample.csv           # columns: ID, transcription
├── train_audio/         # .wav / .mp3 files referenced by train.csv
└── test_audio/          # .wav / .mp3 files referenced by test.csv
```

If your CSVs already contain an absolute/relative path column, set
`AUDIO_PATH_COL` accordingly; otherwise the notebook will build the path as
`{AUDIO_DIR}/{ID}.wav`.


In [5]:

CFG = dict(
    # --- paths (matches the dataset layout) ---
    train_csv       = "/kaggle/input/uyghur-asr/train.csv",
    test_csv        = "/kaggle/input/uyghur-asr/test.csv",
    sample_csv      = "/kaggle/input/uyghur-asr/sample.csv",
    audio_dir       = "/kaggle/input/uyghur-asr/",   # single folder, shared by train + test
    output_dir      = "mms300m-uyghur-ctc",
    submission_path = "submission.csv",

    # column names in the CSVs (train.csv: ID, filepath, transcription | test.csv: ID, filepath)
    id_col          = "ID",
    text_col        = "transcription",
    audio_path_col  = "filepath",   # relative path within wavs/, given directly by the CSVs

    # --- model / training ---
    model_name      = "facebook/mms-300m",
    sampling_rate   = 16000,
    num_epochs      = 4,
    train_batch_size= 1,
    eval_batch_size = 1,
    grad_accum_steps= 1,       # effective batch size = 1 (per-device batch 1, no accumulation)
    learning_rate   = 3e-5,
    warmup_ratio    = 0.1,
    weight_decay    = 0.005,
    fp16            = torch.cuda.is_available(),
    val_size        = 0.05,    # held-out fraction for CER tracking
    max_audio_secs  = 20,      # drop clips longer than this (memory safety)
    freeze_feature_encoder = False,  # False => full fine-tune (recommended here)
)

os.makedirs(CFG["output_dir"], exist_ok=True)
CFG


{'train_csv': '/kaggle/input/uyghur-asr/train.csv',
 'test_csv': '/kaggle/input/uyghur-asr/test.csv',
 'sample_csv': '/kaggle/input/uyghur-asr/sample.csv',
 'audio_dir': '/kaggle/input/uyghur-asr/',
 'output_dir': 'mms300m-uyghur-ctc',
 'submission_path': 'submission.csv',
 'id_col': 'ID',
 'text_col': 'transcription',
 'audio_path_col': 'filepath',
 'model_name': 'facebook/mms-300m',
 'sampling_rate': 16000,
 'num_epochs': 4,
 'train_batch_size': 1,
 'eval_batch_size': 1,
 'grad_accum_steps': 1,
 'learning_rate': 3e-05,
 'warmup_ratio': 0.1,
 'weight_decay': 0.005,
 'fp16': True,
 'val_size': 0.05,
 'max_audio_secs': 20,
 'freeze_feature_encoder': False}

## 4. Load metadata

In [6]:
train_df = pd.read_csv(CFG["train_csv"])
test_df  = pd.read_csv(CFG["test_csv"])
sample_df = pd.read_csv(CFG["sample_csv"])

print(train_df.shape, test_df.shape, sample_df.shape)
train_df.head()


(7574, 3) (1894, 2) (1894, 2)


,ID,filepath,transcription
0,aca14f95f3c4487fa955a9360e324ca5,wavs/aca14f95f3c4487fa955a9360e324ca5.wav,vuyGur HAlqiniN fevudal bAglArniN wA pomexcikl...
1,7f61bb1206384a468f93d36eb4409cfd,wavs/7f61bb1206384a468f93d36eb4409cfd.wav,juNgo HAlqi tOtni zamaniwilaxturux nixaniGa qa...
2,203c32b18c1340b3a99d892651e01367,wavs/203c32b18c1340b3a99d892651e01367.wav,hazirqi sipirliq vapparatlarniN sUrAt hasil qi...
3,d43eb09bfba64eb79f71f7a1bc7eeb03,wavs/d43eb09bfba64eb79f71f7a1bc7eeb03.wav,kambodJadiki wAtAnpArwAr kUclAr wyetnam hOkUmi...
4,554a7ac91a904362a862163804e71b0f,wavs/554a7ac91a904362a862163804e71b0f.wav,bu yArniN vomumiy vuzunluqi tAHminAn miN kilom...


In [7]:
def resolve_audio_path(row, audio_dir, id_col, audio_path_col):
    if audio_path_col is not None and audio_path_col in row and pd.notna(row[audio_path_col]):
        p = str(row[audio_path_col])
        return p if os.path.isabs(p) or os.path.exists(p) else os.path.join(audio_dir, p)
    # fall back: try common extensions using the ID
    for ext in (".wav", ".mp3", ".flac", ".ogg"):
        candidate = os.path.join(audio_dir, f"{row[id_col]}{ext}")
        if os.path.exists(candidate):
            return candidate
    # default guess (wav) even if not found yet — will raise later if truly missing
    return os.path.join(audio_dir, f"{row[id_col]}.wav")

train_df["audio_path"] = train_df.apply(
    lambda r: resolve_audio_path(r, CFG["audio_dir"], CFG["id_col"], CFG["audio_path_col"]),
    axis=1,
)
test_df["audio_path"] = test_df.apply(
    lambda r: resolve_audio_path(r, CFG["audio_dir"], CFG["id_col"], CFG["audio_path_col"]),
    axis=1,
)

missing_train = (~train_df["audio_path"].apply(os.path.exists)).sum()
missing_test  = (~test_df["audio_path"].apply(os.path.exists)).sum()
print(f"Missing train audio files: {missing_train} / {len(train_df)}")
print(f"Missing test audio files:  {missing_test} / {len(test_df)}")


Missing train audio files: 0 / 7574
Missing test audio files:  0 / 1894


## 5. Text normalization

The transcripts in this dataset are in a Latin-script romanization of Uyghur. We keep normalization
light — just Unicode NFC normalization, whitespace collapsing, and stripping
characters that clearly aren't part of the script or basic punctuation the
CTC vocab should model (we let the vocab step decide what's "real"). We
avoid aggressive stripping of diacritics since that can change meaning.


In [8]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

train_df[CFG["text_col"]] = train_df[CFG["text_col"]].apply(normalize_text)
train_df = train_df[train_df[CFG["text_col"]].str.len() > 0].reset_index(drop=True)
train_df.head()


,ID,filepath,transcription,audio_path
0,aca14f95f3c4487fa955a9360e324ca5,wavs/aca14f95f3c4487fa955a9360e324ca5.wav,vuyGur HAlqiniN fevudal bAglArniN wA pomexcikl...,/kaggle/input/uyghur-asr/w...
1,7f61bb1206384a468f93d36eb4409cfd,wavs/7f61bb1206384a468f93d36eb4409cfd.wav,juNgo HAlqi tOtni zamaniwilaxturux nixaniGa qa...,/kaggle/input/uyghur-asr/w...
2,203c32b18c1340b3a99d892651e01367,wavs/203c32b18c1340b3a99d892651e01367.wav,hazirqi sipirliq vapparatlarniN sUrAt hasil qi...,/kaggle/input/uyghur-asr/w...
3,d43eb09bfba64eb79f71f7a1bc7eeb03,wavs/d43eb09bfba64eb79f71f7a1bc7eeb03.wav,kambodJadiki wAtAnpArwAr kUclAr wyetnam hOkUmi...,/kaggle/input/uyghur-asr/w...
4,554a7ac91a904362a862163804e71b0f,wavs/554a7ac91a904362a862163804e71b0f.wav,bu yArniN vomumiy vuzunluqi tAHminAn miN kilom...,/kaggle/input/uyghur-asr/w...


## 6. Build a character-level vocabulary for the CTC tokenizer

In [9]:
def build_vocab(transcripts: List[str]) -> Dict[str, int]:
    vocab_set = set()
    for t in transcripts:
        vocab_set.update(list(t))
    vocab_list = sorted(vocab_set)

    vocab_dict = {ch: i for i, ch in enumerate(vocab_list)}
    # CTC blank/pad + unk + word delimiter, following the standard wav2vec2 recipe
    vocab_dict["|"] = vocab_dict.get(" ", len(vocab_dict))
    if " " in vocab_dict:
        del vocab_dict[" "]
    # re-index cleanly
    ordered_keys = sorted(vocab_dict, key=lambda k: vocab_dict[k])
    vocab_dict = {k: i for i, k in enumerate(ordered_keys)}
    vocab_dict["[UNK]"] = len(vocab_dict)
    vocab_dict["[PAD]"] = len(vocab_dict)
    return vocab_dict

vocab_dict = build_vocab(train_df[CFG["text_col"]].tolist())
print(f"Vocab size: {len(vocab_dict)}")

vocab_path = os.path.join(CFG["output_dir"], "vocab.json")
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)
vocab_dict


Vocab size: 36


{'|': 0,
 'A': 1,
 'G': 2,
 'H': 3,
 'J': 4,
 'N': 5,
 'O': 6,
 'U': 7,
 'a': 8,
 'b': 9,
 'c': 10,
 'd': 11,
 'e': 12,
 'f': 13,
 'g': 14,
 'h': 15,
 'i': 16,
 'j': 17,
 'k': 18,
 'l': 19,
 'm': 20,
 'n': 21,
 'o': 22,
 'p': 23,
 'q': 24,
 'r': 25,
 's': 26,
 't': 27,
 'u': 28,
 'v': 29,
 'w': 30,
 'x': 31,
 'y': 32,
 'z': 33,
 '[UNK]': 34,
 '[PAD]': 35}

## 7. Processor (tokenizer + feature extractor)

In [10]:
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_path,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=CFG["sampling_rate"],
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(CFG["output_dir"])


['mms300m-uyghur-ctc/processor_config.json']

## 8. Build HF `Dataset`s with audio decoding

In [11]:
def to_hf_dataset(df: pd.DataFrame, has_labels: bool) -> Dataset:
    cols = {"audio": df["audio_path"].tolist(), CFG["id_col"]: df[CFG["id_col"]].tolist()}
    if has_labels:
        cols["text"] = df[CFG["text_col"]].tolist()
    ds = Dataset.from_dict(cols)
    ds = ds.cast_column("audio", Audio(sampling_rate=CFG["sampling_rate"]))
    return ds

full_train_ds = to_hf_dataset(train_df, has_labels=True)
test_ds       = to_hf_dataset(test_df, has_labels=False)

split = full_train_ds.train_test_split(test_size=CFG["val_size"], seed=SEED)
train_ds, val_ds = split["train"], split["test"]
print(train_ds, val_ds, test_ds)


Dataset({
    features: ['audio', 'ID', 'text'],
    num_rows: 7195
}) Dataset({
    features: ['audio', 'ID', 'text'],
    num_rows: 379
}) Dataset({
    features: ['audio', 'ID'],
    num_rows: 1894
})


## 9. Preprocessing: audio → input_values, text → labels

In [12]:
def prepare_batch(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    if "text" in batch:
        batch["labels"] = processor.tokenizer(batch["text"]).input_ids   # <-- changed
    return batch

train_ds = train_ds.map(prepare_batch, remove_columns=train_ds.column_names, num_proc=1)
val_ds   = val_ds.map(prepare_batch, remove_columns=val_ds.column_names, num_proc=1)

max_len = CFG["max_audio_secs"] * CFG["sampling_rate"]
before = len(train_ds)
train_ds = train_ds.filter(lambda b: b["input_length"] <= max_len)
print(f"Filtered {before - len(train_ds)} overly long clips from train (> {CFG['max_audio_secs']}s)")


Map (num_proc=1):   0%|          | 0/7195 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/379 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7195 [00:00<?, ? examples/s]

Filtered 322 overly long clips from train (> 20s)


## 10. Data collator for CTC (dynamic padding)

In [13]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")

        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")   # <-- changed

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 11. CER metric

In [14]:
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}


## 12. Load `facebook/mms-300m` for CTC and configure for full fine-tuning

In [15]:
model = Wav2Vec2ForCTC.from_pretrained(
    CFG["model_name"],
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.1,
    mask_time_prob=0.05,
    layerdrop=0.05,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,  # LM head is re-initialized for our vocab
)

if CFG["freeze_feature_encoder"]:
    model.freeze_feature_encoder()
else:
    # Full fine-tune: make sure nothing is frozen
    for p in model.parameters():
        p.requires_grad = True

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {n_trainable:,} / {n_total:,}")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 315,477,670 / 315,477,670


## 13. Training arguments — keep the best checkpoint by CER

In [16]:
training_args = TrainingArguments(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["train_batch_size"],
    per_device_eval_batch_size=CFG["eval_batch_size"],
    gradient_accumulation_steps=CFG["grad_accum_steps"],
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=CFG["num_epochs"],
    fp16=CFG["fp16"],
    gradient_checkpointing=False,
    learning_rate=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"],
    warmup_ratio=CFG["warmup_ratio"],
    save_total_limit=3,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to=[],
    dataloader_num_workers=4,
    push_to_hub=False,
)


trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor.feature_extractor,   # <-- renamed from tokenizer=
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 14. Train

In [17]:
train_result = trainer.train()
trainer.save_model(CFG["output_dir"])
processor.save_pretrained(CFG["output_dir"])
print(train_result)


Epoch,Training Loss,Validation Loss,Cer
1,0.601892,0.319737,0.087560
2,0.368521,0.216380,0.063326
3,0.283047,0.189805,0.053156
4,0.238935,0.173336,0.048840


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=27492, training_loss=0.7658399976741169, metrics={'train_runtime': 7325.9323, 'train_samples_per_second': 3.753, 'train_steps_per_second': 3.753, 'total_flos': 6.979961181118888e+18, 'train_loss': 0.7658399976741169, 'epoch': 4.0})


## 15. Final validation CER

In [18]:
eval_metrics = trainer.evaluate()
print(f"Validation CER: {eval_metrics['eval_cer']:.4f}")


Validation CER: 0.0488


## 16. Inference on `test.csv`

We run batched greedy CTC decoding over the test set. If you want a small
accuracy bump, you can swap this for beam-search decoding with a KenLM
language model via `pyctcdecode`, but greedy decoding is a solid, simple
baseline consistent with how `compute_metrics` above evaluates CER.


In [19]:
import gc

# Free the Trainer's internal state (optimizer, scheduler, duplicate model refs)
del trainer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

1.279998976 GB allocated
1.348468736 GB reserved


In [20]:
@torch.no_grad()
def transcribe_dataset(ds, batch_size=2):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    if device == "cuda":
        model.half()  # fp16 inference

    predictions = []
    for i in range(0, len(ds), batch_size):
        batch = ds[i : i + batch_size]
        audio_arrays = [a["array"] for a in batch["audio"]]
        inputs = processor(
            audio_arrays, sampling_rate=CFG["sampling_rate"], return_tensors="pt", padding=True
        )
        input_values = inputs.input_values.to(device)
        if device == "cuda":
            input_values = input_values.half()
        attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None

        logits = model(input_values, attention_mask=attention_mask).logits
        pred_ids = torch.argmax(logits, dim=-1)
        batch_preds = processor.batch_decode(pred_ids)
        predictions.extend(batch_preds)

        del input_values, logits, pred_ids
        torch.cuda.empty_cache()
    return predictions

test_predictions = transcribe_dataset(test_ds, batch_size=2)
len(test_predictions)

1894

## 17. Build `submission.csv`

In [21]:
submission_df = pd.DataFrame({
    "ID": test_ds[CFG["id_col"]],
    "transcription": test_predictions,
})

# sanity checks against the provided sample format
assert set(submission_df.columns) == {"ID", "transcription"}
assert set(submission_df["ID"]) == set(sample_df["ID"]), "ID mismatch vs sample.csv!"

submission_df = submission_df.set_index("ID").loc[sample_df["ID"]].reset_index()
submission_df.to_csv("submission.csv", index=False)
submission_df.head()


,ID,transcription
0,f068a206b84c4632865e0629a1b62fb8,bu dorini helila qaynatqan caqqan bol vissiqid...
1,a9d8cfab47b34f12b8f4b4769075713e,yamGurdin keyinki hawa Huddi sUzUp tazlanGandA...
2,34147b4f995144288b720d7474ba4dd6,qar barGancA qattiq yaGdi yoldiki piyadilAr te...
3,c6c201bcd81a402385c2f008983f7474,cAtkA ciqip bilim vigAnligAndin keyin qaytip k...
4,c3c190cc67c14d4a946ef1b722196248,vAyiblAx kixini cUxkUnlAxtUridu vilhamlandurux...


## 18. Notes / next steps if CER needs to go lower

- **Beam search + language model**: train a small char/word-level KenLM on
  the training transcripts (plus any extra Uyghur text you can find) and
  decode with `pyctcdecode` — this is usually the single biggest CER win
  after full fine-tuning.
- **SpecAugment tuning**: `mask_time_prob` / `mask_feature_prob` can be
  nudged if you see overfitting after epoch ~5–6.
- **Longer warmup / lower LR**: if loss spikes early with full fine-tuning,
  try `learning_rate=1e-5` to `2e-5` with a longer warmup ratio.
- **Text normalization**: double check Uyghur-specific punctuation/digit
  handling in `normalize_text` — inconsistent normalization between train
  and test text is a common silent source of CER inflation.
- **Checkpoint selection**: since you saw gains only after epoch 2 on the
  first run, `load_best_model_at_end=True` + `metric_for_best_model="cer"`
  above ensures you keep the best of all epochs even if later epochs
  regress.
